# <center>The Assignment Problem</center>
### <center>Alfred Galichon (NYU & Sciences Po), Antoine Jacquet (Sciences Po) and Georgy Salakhutdinov (Polytechnique)</center>
## <center>'math+econ+code' masterclass series, Collegio Carlo Alberto, March 2026</center>
#### <center>With python code examples</center>
© 2018–2026 by Alfred Galichon. Past and present support from NSF grant DMS-1716489, ERC grant CoG-866274 are acknowledged, as well as inputs from contributors listed [here](http://www.math-econ-code.org/team).

**If you reuse material from this masterclass, please cite as:**<br>
Alfred Galichon, 'math+econ+code' masterclass series. https://www.math-econ-code.org/

## Learning objectives

* Assignment problem
* Linear programming and duality
* Solving LPs with Gurobi
* Large-scale assignment with observable types


## References

* Shapley & Shubik (1971). The Assignment Game I: The Core. *International Journal of Game Theory*
* Dantzig (1963). Linear Programming and Extensions. *Princeton University Press*


## Libraries


In [4]:
!pip install gurobipy
from google.colab import files
#uploaded = files.upload()              # pick local file in the browser
#src = next(iter(uploaded))             # uploaded filename, e.g. "gurobi.lic"
#shutil.move(src, "/content/gurobi.lic")

import os
os.environ["GRB_LICENSE_FILE"] = "/content/gurobi.lic"

import numpy as np
import gurobipy as grb


# The Assignment problem

## Setting

Consider a two-sided population of individuals, say women and men, to be matched (could also be workers and jobs, CEOs and firms...).
- Women are indexed by $i \in I$, men by $j \in J$, with $I$ and $J$ finite.
- A match between woman $i$ and man $j$ produces a value $\Phi_{ij}$, called *surplus*.
- Unmatched individuals produce a value normalized to 0.
- A matching is represented by a matrix $\pi = (\pi_{ij})$ whose entries indicate whether $i$ and $j$ are matched.

The social planner's problem is to match (assign) individuals in order to maximize the total surplus in the population, that is,

\begin{align}
\max_{\pi_{ij} \geq 0} ~ & \sum_{ij} \pi_{ij} \Phi_{ij} \tag{A} \\
\text{s.t.} ~            & \textstyle\sum_j \pi_{ij} \leq 1 \qquad (\forall i) \\
                         & \textstyle\sum_i \pi_{ij} \leq 1 \qquad (\forall j).
\end{align}

The constraints are simply feasibility constraints on the match: any individual can be matched to at most one other individual.

The problem above is a **linear program** (or LP): an optimization problem where both the objective and the constraints are linear in the optimizing variables.


## Numerical example

We begin by solving a simple assignment problem using Gurobi.

In [5]:
np.random.seed(0)
I, J = 12, 10
Phi_i_j = np.random.uniform(size = (I,J))

m = grb.Model()
pi_i_j = m.addMVar((I,J))
m.setObjective( (pi_i_j * Phi_i_j).sum(), grb.GRB.MAXIMIZE)
m.addConstr( pi_i_j.sum(axis=1) <= 1.0 )
m.addConstr( pi_i_j.sum(axis=0) <= 1.0 )
m.optimize()
pi_i_j = np.array(m.x).reshape((I,J))
print(pi_i_j)
print((pi_i_j * Phi_i_j).sum())

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2790720
Academic license 2790720 - for non-commercial use only - registered to an___@sciencespo.fr
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Academic license 2790720 - for non-commercial use only - registered to an___@sciencespo.fr
Optimize a model with 22 rows, 120 columns and 240 nonzeros (Max)
Model fingerprint: 0x4fef2bff
Model has 120 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [5e-03, 1e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+00]

Presolve time: 0.01s
Presolved: 22 rows, 120 columns, 240 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    5.9627023e+31   2.400000e+32   5.962702e+01    

## A crash course on linear programming

Before we analyze the assignment problem further, we need to introduce some important concepts from linear programming.

A linear problem in canonical form takes the form
\begin{align}
\max_{x \in \mathbb R_+^n} ~ & c^\top x \tag{P} \\
\text{s.t.} ~ & Ax \leq b
\end{align}
where $A$ is an $n \times m$ matrix, $c \in \mathbb R^n$ and $b \in \mathbb R^m$.
For reasons which will become clear soon, this program is called the **primal problem**.

---

**Minimax inequality.**
For any function $f : \mathbb R_+^n \times \mathbb R_+^m \to \mathbb R$, we have:

\begin{equation}
\max_{x \in \mathbb R_+^n} \min_{y \in \mathbb R_+^m} ~ f(x,y) \leq \min_{y \in \mathbb R_+^m} \max_{x \in \mathbb R_+^n} ~ f(x,y).
\end{equation}

*Proof.*
For any $x$ and $y$, $\min_{\tilde y \in \mathbb R_+^m} ~ f(x, \tilde y) \leq f(x,y) \leq \max_{\tilde x \in \mathbb R_+^n} ~ f(\tilde x,y)$,
hence the result.

---

Any linear program admits a dual problem, which is obtained by forming the Lagrangian and using the minimax inequality:

\begin{align}
\max_{x \in \mathbb R_+^n} ~ \bigg\{ \sum_j c_j x_j \quad \text{s.t.} ~ Ax \leq b \bigg\}
=    &\max_{x \in \mathbb R_+^n} ~ \min_{y \in \mathbb R_+^m} ~ \bigg\{ \sum_j c_j x_j + \sum_i y_i (b - Ax)_i \bigg\} \\
\leq &\min_{y \in \mathbb R_+^m} ~ \max_{x \in \mathbb R_+^n} ~ \bigg\{ \sum_j c_j x_j + \sum_i b_i y_i + \sum_{ij} x_j A_{ij} y_i \bigg\} \\
=    &\min_{y \in \mathbb R_+^m} ~ \max_{x \in \mathbb R_+^n} ~ \bigg\{ \sum_i b_i y_i + \sum_j x_j (c - A^\top y)_j \bigg\} \\
\end{align}

which finally yields the **dual problem**

\begin{align}
\min_{y \in \mathbb R_+^m} ~ & b^\top y \tag{D} \\
\text{s.t.} ~ & A^\top y \geq c.
\end{align}

In general the value of the dual problem $(\text{D})$ is weakly greater than that of the primal problem $(P)$; however, it can be shown that if one of them has a finite value, then both values coincide and we have the following optimality conditions:

**KKT optimality conditions for LPs.**
- Primal feasibility: $x \geq 0$ and $Ax \leq b$,
- Dual feasibility: $y \geq 0$ and $A^\top y \geq c$,
- Complementary slackness: $y_i (b - Ax)_i = 0$ for all $i$ and $x_j (A^\top y - c)_j = 0$ for all $j$.


## Competitive equilibrium interpretation

Now let's derive the dual of our assignment problem $(\text{A})$:
\begin{align}
\max_{\pi_{ij} \geq 0} ~ \bigg\{ \sum_{ij} \pi_{ij} \Phi_{ij} \quad \text{s.t.} ~ \sum_j \pi_{ij} \leq 1, \sum_i \pi_{ij} \leq 1 \bigg\}
=    &\max_{\pi_{ij} \geq 0} ~ \min_{u_i, v_j \geq 0} ~ \bigg\{ \sum_{ij} \pi_{ij} \Phi_{ij}
           + \sum_i u_i (1 - \sum_j \pi_{ij}) + \sum_j v_j (1 - \sum_i \pi_{ij})  \bigg\} \\
=    &\min_{u_i, v_j \geq 0} ~ \max_{\pi_{ij} \geq 0} ~ \bigg\{ \sum_i u_i + \sum_j v_j + \sum_{ij} \pi_{ij} (\Phi_{ij} - u_i - v_j)  \bigg\}
\end{align}
hence its dual is
\begin{align}
\min_{u_i, v_j \geq 0} ~ & \sum_i u_i + \sum_j v_j \\
\text{s.t.} ~ & u_i + v_j \geq \Phi_{ij} \qquad (\forall ij).
\end{align}

We deduct the KKT optimality conditions for the assignment problem: they are values of the primal variables $\pi_{ij}$ and dual variables $u_i$ and $v_j$ such that

- Primal feasibility:  
  $\pi_{ij} \geq 0$, $\sum_j \pi_{ij} \leq 1$, $\sum_i \pi_{ij} \leq 1$
- Dual feasibility:  
  $u_i \geq 0$, $v_j \geq 0$, $u_i + v_j \geq \Phi_{ij}$
- Complementary slackness:  
  $\pi_{ij} > 0 \implies u_i + v_j \geq \Phi_{ij}, \quad \sum_j \pi_{ij} < 1 \implies u_i = 0, \quad \sum_i \pi_{ij} < 1 \implies v_j = 0$.

Shapley and Shubik (1971) noted that these conditions correspond exactly to those of a *competitive equilibrium* in a decentralized market with transferable utility (TU), where $u_i$ and $v_j$ play the role of the agents' utilities:
- As before, each matched pair $ij$ generates surplus $\Phi_{ij}$.
- This surplus can be split arbitrarily between two matched agents:
if woman $i$ and man $j$ are matched with payoffs $u_i$ and $v_j$ respectively, then $u_i + v_j = \Phi_{ij}$.
- Unmatched individuals receive utility normalized to 0.

Indeed:
- Primal feasibility corresponds to feasibility constraints on the matching $\pi$.
- Dual feasibility corresponds to individual rationality and stability, i.e. no pair $ij$ can jointly obtain more surplus by matching together than what they already receive.
- Complementary slackness corresponds to feasibility constraints on the payoffs $u$ and $v$.

<!--
**Definition.** A *competitive equilibrium* consists of a matching $\pi = (\pi_{ij})$ and a payoff vectors $u = (u_i)$ and $v = (v_j)$ which satisfy the following:
- $\pi_{ij} \geq 0$, $\sum_j \pi_{ij} \leq 1$, $\sum_i \pi_{ij} \leq 1$ (Matching feasibility)  
- $u_i \geq 0$, $v_j \geq 0$, $u_i + v_j \geq \Phi_{ij}$ (Stability)  
- $\pi_{ij} > 0 \implies u_i + v_j \geq \Phi_{ij}, \quad \sum_j \pi_{ij} < 1 \implies u_i = 0, \quad \sum_i \pi_{ij} < 1 \implies v_j = 0$ (Payoff feasibility).

The second condition in particular means that no pair $ij$ can jointly obtain more surplus by matching together than what they already receive.
-->


# Assignment problem with separability

Now assume that the population of individuals to match is very large and is also distributed on observable types.  
Women are sorted into types $x \in X$ and men into types $y \in Y$, with $x_i$ and $y_j$ denoting the types of individuals $i$ and $j$.

We make a crucial assumption on the surplus produced by $i$ and $j$ in this context.

---

**Separability assumption.** (Choo & Siow 2006; Chiappori, Salanié & Weiss 2017)  
The surplus $\tilde \Phi_{ij}$ produced by $i$ and $j$ is the sum of a systematic part $\Phi_{xy}$ which only depends on the types $x_i = x$ and $y_j = y$ of the matched agents, and idiosyncratic preference shocks $\varepsilon_{iy}$ and $\eta_{xj}$ of the agents over potential partner types:
\begin{equation}
\tilde \Phi_{ij} = \Phi_{xy} + \varepsilon_{iy} + \eta_{xj}.
\end{equation}
Furthermore, the utility obtained from remaining single is $\varepsilon_{i0}$ for woman $i$ and $\eta_{0j}$ for man $j$.
Finally, the idiosyncratic preference shocks $\varepsilon_{iy}$ and $\eta_{xj}$ follow distributions conditional on the individuals' type:
$\varepsilon_{iy} \, | \, x_i = x \; \sim \; \mathbf P_x$ and $\eta_{xj} \, | \, y_j = y \; \sim \; \mathbf Q_y$.

---

Note that the separability assumption implies that individuals are indifferent across partners of the same observable type.


## Reformulation of the assignment problem

An important consequence of the separability assumption is that we don't need to track anymore exactly which individual $i$ matches with which individual $j$.
Instead, it is enough to only track which type each individual is matched with.
This naturally motivates the introduction of the following aggregate matching variables:
\begin{equation}
    \pi_{iy} = \sum_{j : y_j = y} \tilde \pi_{ij} \quad \text{and} \quad \pi_{xj} = \sum_{i : x_i = x} \tilde \pi_{ij}.
\end{equation}

Consistency requires that the mass of women of type $x$ matched with men of type $y$ equals the mass of men of type $y$ matched with women of type $x$:
\begin{equation}
    \sum_{i : x_i = x} \pi_{iy} = \sum_{j : y_j = y} \pi_{xj}.
\end{equation}

With the separability assumption, the objective of the assignment problem $(A)$ becomes

\begin{align}
\max_{\pi_{iy}, \pi_{i0}, \pi_{xj}, \pi_{0j} \geq 0} ~ & \sum_i \bigg[ \pi_{i0} \varepsilon_{i0} + \sum_{y \in Y} \pi_{iy} \alpha_{iy} \bigg]
+ \sum_j \bigg[ \pi_{0j} \eta_{0j} + \sum_{x \in X} \pi_{xj} \gamma_{xj} \bigg] \\
\text{s.t.} ~ & \pi_{i0} + \textstyle\sum_{y \in Y} \pi_{iy} = 1 \qquad (\forall i) \notag \\
              & \pi_{0j} + \textstyle\sum_{x \in X} \pi_{xj} = 1 \qquad (\forall j) \notag \\
              & \textstyle\sum_{i : x_i = x} \pi_{iy} = \textstyle\sum_{j : y_j = y} \pi_{xj}  \quad (\forall xy) \notag.
\end{align}

where

\begin{equation}
    \alpha_{iy} = \frac{\Phi_{x_i y}}{2} + \varepsilon_{iy}
    \quad \text{and} \quad
    \gamma_{xj} = \frac{\Phi_{x y_j}}{2} + \eta_{xj}
\end{equation}

are the default shares of the surplus accruing to woman $i$ when matched with a type-$y$ man and to man $j$ when matched with a type-$x$ woman, respectively.

This new version of the problem has $I (Y+1) + J (X+1)$ variables and $XY + I + J$ constraints, compared to $IJ + I + J$ variables and $I + J$ constraints in the assignment problem with general surplus $\tilde \Phi_{ij}$.

We implement this approach below.


In [6]:
import time

def solveSeparableAssignment(eps_i_y, eps_i_0, eta_x_j, eta_0_j, Phi_x_y, delta_i_x, delta_j_y, verbose=0):
    # delta_i_x is an indicator vector equal to 1 iff x_i = x (idem for delta_j_y)
    t_start = time.perf_counter()

    I, Y = eps_i_y.shape
    X, J = eta_x_j.shape
    phi_i_y = delta_i_x @ Phi_x_y / 2 + eps_i_y
    psi_x_j = Phi_x_y @ delta_j_y.T / 2 + eta_x_j

    m = grb.Model()
    if verbose==0: m.Params.OutputFlag = 0
    pi_i_y = m.addMVar((I,Y), lb=0.0)
    pi_x_j = m.addMVar((X,J), lb=0.0)
    pi_i_0 = m.addMVar(I, lb=0.0)
    pi_0_j = m.addMVar(J, lb=0.0)
    obj = pi_i_0.T @ eps_i_0 + pi_0_j.T @ eta_0_j + (pi_i_y * phi_i_y).sum() + (pi_x_j * psi_x_j).sum()
    m.setObjective(obj, grb.GRB.MAXIMIZE)
    m.addConstr( pi_i_0 + pi_i_y.sum(axis=1) == 1.0 )
    m.addConstr( pi_0_j + pi_x_j.T.sum(axis=1) == 1.0 )
    m.addConstr( delta_i_x.T @ pi_i_y == pi_x_j @ delta_j_y )
    m.optimize()

    pi = np.array(m.x[:I*Y+X*J])
    pi_i_y = pi[:I*Y].reshape((I,Y))
    pi_x_j = pi[I*Y:].reshape((X,J))

    t = time.perf_counter() - t_start
    print("\nTotal time:", t)

    return pi_i_y, pi_x_j


In [7]:
X, Y = 40, 30
np.random.seed(0)
Phi_x_y = np.random.normal(0, 5, size=(X, Y))

# Small example
I, J = 500, 400
x_i = np.random.randint(0, X, size=I)
y_j = np.random.randint(0, Y, size=J)
delta_i_x = np.eye(X)[x_i]
delta_j_y = np.eye(Y)[y_j]
eps_i_y = np.random.normal(0, 0.1, size=(I, Y))
eta_x_j = np.random.normal(0, 0.1, size=(X, J))
eps_i_0 = np.random.normal(0, 0.1, size=I)
eta_0_j = np.random.normal(0, 0.1, size=J)
phi_i_y = delta_i_x @ Phi_x_y / 2 + eps_i_y
psi_x_j = Phi_x_y @ delta_j_y.T / 2 + eta_x_j

print("Small example (I=500, J=400, X=40, Y=30)")
solveSeparableAssignment(eps_i_y, eps_i_0, eta_x_j, eta_0_j, Phi_x_y, delta_i_x, delta_j_y, verbose=0)


print("\n---\n")

# Large example (scaled by 16)
I, J = 8000, 6400
x_i = np.random.randint(0, X, size=I)
y_j = np.random.randint(0, Y, size=J)
delta_i_x = np.eye(X)[x_i]
delta_j_y = np.eye(Y)[y_j]
eps_i_y = np.random.normal(0, 0.1, size=(I, Y))
eta_x_j = np.random.normal(0, 0.1, size=(X, J))
eps_i_0 = np.random.normal(0, 0.1, size=I)
eta_0_j = np.random.normal(0, 0.1, size=J)
phi_i_y = delta_i_x @ Phi_x_y / 2 + eps_i_y
psi_x_j = Phi_x_y @ delta_j_y.T / 2 + eta_x_j

print("Large example (I=8000, J=6400, X=40, Y=30)")
solveSeparableAssignment(eps_i_y, eps_i_0, eta_x_j, eta_0_j, Phi_x_y, delta_i_x, delta_j_y, verbose=0)

Small example (I=500, J=400, X=40, Y=30)

Total time: 2.8983574970000063

---

Large example (I=8000, J=6400, X=40, Y=30)

Total time: 68.36970376400006


(array([[0., 0., 0., ..., 1., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]]),
 array([[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]]))

In practice, the complexity of this method seems to be between $O(s^{3/2})$ and $O(s^2)$ where $s$ is the scale of the population ($|I| = s$ and $|J| = s$), and is increasing in the size of the type sets $X$ and $Y$. (See Galichon, Jacquet and Salakhutdinov 2025).